# Pipeline: ATE + ASC (Two-Stage) & Error Analysis

Ket hop 2 model tot nhat:
1. **ATE model** tim aspect spans tu cau van
2. **ASC model** phan loai cam xuc cho tung span

Sau do so sanh voi Ground Truth de lam **Error Analysis**.


## 1. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import load_preprocessed, tokenize_baseline
from src.ate.ate_dataset import ATEDataset, BIO_TAGS, TAG2ID, NUM_TAGS, ASPECTS
from src.ate.ate_model import build_ate_model
from src.asc.asc_model import build_asc_model
from src.utils.engine import predict_ate, predict_asc
from src.utils.metrics import bio_tags_to_spans, evaluate_spans_f1, token_accuracy
from src.utils.visualization import plot_error_analysis

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Load Preprocessed Data & Best Models


In [ ]:
# Load shared vocab & data
PREP_DIR = os.path.join("..", "..", "preprocessed")
word2idx, emb_matrix, train_items, dev_items, test_items = load_preprocessed(PREP_DIR)
VOCAB_SIZE = len(word2idx)
EMB_DIM = emb_matrix.shape[1]

# === THAY DOI TEN MODEL THEO KET QUA CUA NOTEBOOK 01 & 02 ===
# Doc file CSV de biet model nao tot nhat
ate_results = pd.read_csv("../../results/ate/ate_results.csv")
asc_results = pd.read_csv("../../results/asc/asc_results.csv")
best_ate_name = ate_results.loc[ate_results["Span_F1"].idxmax(), "Model"]
best_asc_name = asc_results.loc[asc_results["Macro_F1"].idxmax(), "Model"]
print(f"Best ATE: {best_ate_name}")
print(f"Best ASC: {best_asc_name}")

# Load ATE model
ate_type = best_ate_name.replace("-CRF", "")
ate_model = build_ate_model(
    model_type=ate_type, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
    hidden_dim=256, num_tags=NUM_TAGS, n_layers=2, dropout=0.3,
).to(device)
ate_model.load_state_dict(torch.load(f"../../results/ate/best_ate_{best_ate_name}.pt", map_location=device))
ate_model.eval()
print(f"ATE model loaded: {best_ate_name}")

# Load ASC model
asc_model = build_asc_model(
    model_type=best_asc_name, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
    hidden_dim=256, num_classes=3, n_layers=2, dropout=0.3,
).to(device)
asc_model.load_state_dict(torch.load(f"../../results/asc/best_asc_{best_asc_name}.pt", map_location=device))
asc_model.eval()
print(f"ASC model loaded: {best_asc_name}")


## 3. Pipeline Inference
Buoc 1: ATE predict spans -> Buoc 2: ASC classify sentiment cho moi span.


In [ ]:
MAX_LEN = 128
SENTIMENT_NAMES = {0: "POSITIVE", 1: "NEGATIVE", 2: "NEUTRAL"}

def pipeline_predict(text, ate_model, asc_model, word2idx, device):
    """Chay pipeline tren 1 cau: ATE -> ASC"""
    words = text.split()[:MAX_LEN]

    # Step 1: ATE - predict aspect spans
    seq, length = tokenize_baseline(text, word2idx, MAX_LEN)
    seq_t = torch.tensor([seq], dtype=torch.long).to(device)
    mask_t = torch.zeros(1, MAX_LEN, dtype=torch.bool)
    mask_t[0, :length] = True
    mask_t = mask_t.to(device)
    lens_t = torch.tensor([length])

    with torch.no_grad():
        pred_tags = ate_model(seq_t, mask=mask_t, lens=lens_t)[0]

    spans = bio_tags_to_spans(pred_tags, BIO_TAGS, length)

    # Step 2: ASC - classify sentiment for each span
    results = []
    for aspect_label, start_idx, end_idx in spans:
        # Boc aspect bang [ASP]
        aspect_text = " ".join(words[start_idx:end_idx])
        marked = " ".join(words[:start_idx]) + " [ASP] " + aspect_text + " [ASP] " + " ".join(words[end_idx:])

        asc_seq, _ = tokenize_baseline(marked.strip(), word2idx, MAX_LEN)
        asc_t = torch.tensor([asc_seq], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = asc_model(asc_t)
            sentiment_id = logits.argmax(dim=1).item()

        results.append({
            "aspect_text": aspect_text,
            "aspect_category": aspect_label,
            "sentiment": SENTIMENT_NAMES[sentiment_id],
            "combined_label": f"{aspect_label}#{SENTIMENT_NAMES[sentiment_id]}"
        })

    return results

# Test thu 5 cau
print("=== Pipeline Prediction Demo ===")
for i in range(min(5, len(test_items))):
    text = test_items[i]["text"]
    preds = pipeline_predict(text, ate_model, asc_model, word2idx, device)
    true_labels = [label for _, _, label in test_items[i].get("labels", [])]
    print(f"\nText: {text[:80]}...")
    print(f"  True: {true_labels}")
    print(f"  Pred: {[p['combined_label'] for p in preds]}")


## 4. Full Test Set Evaluation


In [ ]:
print("Running pipeline on full test set...")
all_true_labels = []
all_pred_labels = []

for item in test_items:
    text = item["text"]
    true_set = set(label for _, _, label in item.get("labels", []))
    preds = pipeline_predict(text, ate_model, asc_model, word2idx, device)
    pred_set = set(p["combined_label"] for p in preds)

    all_true_labels.append(true_set)
    all_pred_labels.append(pred_set)

# Tinh Micro/Macro F1 tren tap label sets
tp, fp, fn = 0, 0, 0
for true_set, pred_set in zip(all_true_labels, all_pred_labels):
    tp += len(true_set & pred_set)
    fp += len(pred_set - true_set)
    fn += len(true_set - pred_set)

p = tp / (tp + fp) if (tp + fp) > 0 else 0
r = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"\n=== PIPELINE END-TO-END RESULTS ===")
print(f"  Precision: {p:.4f}")
print(f"  Recall:    {r:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  (TP={tp}, FP={fp}, FN={fn})")


## 5. Error Analysis
Phan tich chi tiet nhung cau bi sai: False Positive va False Negative.


In [ ]:
errors = []
for i, (true_set, pred_set) in enumerate(zip(all_true_labels, all_pred_labels)):
    fp_labels = pred_set - true_set
    fn_labels = true_set - pred_set

    if fp_labels or fn_labels:
        for label in fn_labels:
            errors.append({
                "Text": test_items[i]["text"][:100],
                "Error_Type": f"MISSED: {label}",
                "Category": label.split("#")[0] if "#" in label else label,
                "Direction": "False Negative"
            })
        for label in fp_labels:
            errors.append({
                "Text": test_items[i]["text"][:100],
                "Error_Type": f"WRONG: {label}",
                "Category": label.split("#")[0] if "#" in label else label,
                "Direction": "False Positive"
            })

error_df = pd.DataFrame(errors)
print(f"Tong so loi: {len(error_df)}")
print(f"  False Negatives: {(error_df['Direction']=='False Negative').sum()}")
print(f"  False Positives: {(error_df['Direction']=='False Positive').sum()}")

# Top loi theo Category
print("\n=== LOI THEO ASPECT CATEGORY ===")
display(error_df.groupby(["Category", "Direction"]).size().unstack(fill_value=0).sort_values(
    "False Negative", ascending=False))


## 6. Error Visualization


In [ ]:
# Bieu do loi
plot_error_analysis(error_df, top_n=15)

# Xem 10 cau False Negative dien hinh (model bo sot)
print("\n=== 10 CAU BI BO SOT (FALSE NEGATIVE) ===")
fn_df = error_df[error_df["Direction"] == "False Negative"].head(10)
display(fn_df[["Text", "Error_Type"]])


## 7. Save Error Report


In [ ]:
SAVE_DIR = os.path.join("..", "..", "results", "pipeline")
os.makedirs(SAVE_DIR, exist_ok=True)
error_df.to_csv(os.path.join(SAVE_DIR, "error_analysis.csv"), index=False)
print(f"Pipeline F1: {f1:.4f}")
print(f"Error report saved to {SAVE_DIR}/error_analysis.csv")
print("\nDone! Chuyen sang Notebook 04 de so sanh voi E2E PhoBERT.")
